# Phases 6–7: Explainability and greening scenarios

* `src/mirageai/explain.py`: interventional TreeSHAP, driver groups, threshold finder
* `src/mirageai/scenarios.py`: target cells, tree-cover edits, index models, support guard

Run by `scripts/pipeline/run_phase6.py` (≈30 min of SHAP) and `scripts/pipeline/run_phase7.py`; this notebook reads their outputs.

In [1]:
import pandas as pd
from mirageai import config, results as R

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

## Global importance (mean |SHAP|, °C)

In [2]:
R.load_table("phase6_group_importance")

,group,lgbm_all,lgbm_urban,xgb_all,xgb_urban,lgbm_path_dependent
0,spectral,1.488,1.022,1.489,1.030,1.535
1,dry cover,0.725,0.823,0.739,0.838,0.807
2,tree cover,0.382,0.305,0.346,0.287,0.416
3,terrain,0.293,0.240,0.306,0.256,0.292
4,water proximity,0.168,0.107,0.184,0.123,0.144
5,built,0.125,0.167,0.089,0.090,0.118


In [3]:
R.load_table("phase6_shap_importance")[["feature", "lgbm_all", "lgbm_urban", "xgb_all", "rank_lgbm_all"]]

,feature,lgbm_all,lgbm_urban,xgb_all,rank_lgbm_all
0,NDVI,0.955,0.819,1.098,1
1,NDBI,0.728,0.539,0.588,2
2,MNDWI,0.612,0.797,0.747,3
3,frac_grass,0.541,0.535,0.555,4
4,frac_tree,0.382,0.305,0.346,5
5,frac_crop,0.342,0.289,0.343,6
6,elevation,0.253,0.226,0.261,7
7,slope,0.156,0.124,0.153,8
8,dist_water_m,0.156,0.106,0.171,9
9,frac_built,0.125,0.167,0.089,10


![SHAP beeswarm](../outputs/figures/06_shap_beeswarm.png)

![Dominant driver map](../outputs/figures/06_dominant_driver_map.png)

## Thresholds (D-037)

In [4]:
R.load_table("phase6_thresholds")[["feature", "shape", "knot", "knot_lo", "knot_hi"]]

,feature,shape,knot,knot_lo,knot_hi
0,NDVI,changes slope,0.274,0.266,0.285
1,NDBI,near-linear,0.088,0.084,0.091
2,MNDWI,onset,-0.394,-0.400,-0.391
3,frac_grass,saturates,0.045,0.034,0.055
4,frac_tree,saturates,0.012,0.012,0.012
5,frac_crop,saturates,0.024,0.023,0.024
6,elevation,saturates,590.494,589.794,591.154
7,slope,near-linear,1.371,1.371,2.240
8,dist_water_m,saturates,188.254,184.810,191.698
9,frac_built,saturates,0.094,0.075,0.124


## Greening scenarios (RQ4)

Headline ΔLST in the 2,673 hottest urban cells, mean of three models with the min–max across them,
excluding the 0 → 1% "any canopy" step (D-040). **Associational, not causal** (D-009).

In [5]:
R.scenario_bracket(R.load_table("phase7_summary"))[
    ["scenario_label", "bound", "dLST", "dLST_lo", "dLST_hi", "pct_flagged"]]

,scenario_label,bound,dLST,dLST_lo,dLST_hi,pct_flagged
0,A: +10 points,lower,-0.374,-0.417,-0.329,0.000
1,A: +10 points,realistic,-1.017,-1.045,-1.003,0.012
2,B: +20 points,lower,-0.565,-0.653,-0.486,0.037
3,B: +20 points,realistic,-1.630,-1.672,-1.603,0.037
4,C: raise to the 10.2% knot,lower,-0.350,-0.376,-0.315,0.000
5,C: raise to the 10.2% knot,realistic,-0.967,-1.004,-0.935,0.012


![Scenario maps](../outputs/figures/07_delta_lst_maps.png)